# Citation Cartel Detection: Graph-Based Anomaly Detection

This notebook demonstrates a novel graph-based method for detecting citation cartels in academic networks.

**What is a citation cartel?** A group of journals that artificially inflate citation metrics by excessively citing each other.

**Dataset:** Journal citation network from OpenAlex (2015-2022), 96 top journals, 888 directed citation edges.

**Method:** Combines Hodge decomposition, Louvain community detection, and Isolation Forest anomaly scoring.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

_pip('python-louvain==0.16')
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'networkx==3.6.1')

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import community.community_louvain as community_louvain

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-280814-hodge-decomposition-for-citation-cartel/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'][0]['examples'])} journal examples")
print(f"Dataset: {data['metadata']['description']}")
print(f"Graph stats: {data['metadata']['graph_stats']}")

## Configuration

Tunable parameters — start with minimal values for quick demo.

In [ ]:
N_SAMPLES = 3
ISOLATION_FOREST_CONTAMINATION = 0.2
ISOLATION_FOREST_RANDOM_STATE = 42
LOUVAIN_SEED = 42
MUTUAL_CITATION_RATIO_THRESHOLD = 0.3
N_MUTUAL_PARTNERS_THRESHOLD = 3

print(f"Config: {N_SAMPLES} samples, contamination={ISOLATION_FOREST_CONTAMINATION}")

## Data Preparation

Extract journal features from the dataset.

In [ ]:
examples = data['datasets'][0]['examples'][:N_SAMPLES]
journals = []

for ex in examples:
    feat = json.loads(ex['input'])
    feat['label'] = ex['output']
    feat['journal_id'] = ex['metadata_journal_id']
    journals.append(feat)

df = pd.DataFrame(journals)
print(f"Extracted {len(df)} journals")
print(df[['journal_name', 'in_degree', 'out_degree', 'mutual_citation_weight', 'label']])

## Hodge Decomposition

Decompose the citation network into gradient (asymmetric) and cyclic (mutual) components.

In [ ]:
def compute_hodge_decomposition(df):
    gradient_component = np.abs(df['in_citation_weight'].values - df['out_citation_weight'].values)
    cyclic_component = df['mutual_citation_weight'].values
    return {'gradient': gradient_component, 'cyclic': cyclic_component}

hodge = compute_hodge_decomposition(df)
print(f"Gradient component: {hodge['gradient']}")
print(f"Cyclic component: {hodge['cyclic']}")

## Community Detection

Use Louvain algorithm to identify journal clusters.

In [ ]:
def build_citation_graph(df):
    G = nx.DiGraph()
    for i, row in df.iterrows():
        G.add_node(i, journal_name=row['journal_name'], label=row['label'])
    for i, row in df.iterrows():
        for j in range(len(df)):
            if i != j and row['mutual_citation_weight'] > 0:
                G.add_edge(i, j, weight=row['mutual_citation_weight'])
    return G

G = build_citation_graph(df)
print(f"Citation graph: {len(G.nodes())} nodes, {len(G.edges())} edges")

G_undirected = G.to_undirected()
if len(G_undirected.edges()) > 0:
    partition = community_louvain.best_partition(G_undirected, seed=LOUVAIN_SEED)
    df['community'] = df.index.map(partition)
    print(f"Communities: {len(set(partition.values()))}")
else:
    df['community'] = -1
    print("No edges in graph")

## Anomaly Detection

Use Isolation Forest to score anomalous journals.

In [ ]:
def detect_anomalies(df):
    features = [
        'in_degree', 'out_degree', 'in_citation_weight', 'out_citation_weight',
        'mutual_citation_weight', 'citation_asymmetry', 'mutual_citation_ratio',
        'n_mutual_partners'
    ]
    X = df[features].fillna(0).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    iso_forest = IsolationForest(
        contamination=ISOLATION_FOREST_CONTAMINATION,
        random_state=ISOLATION_FOREST_RANDOM_STATE,
        n_estimators=100
    )
    anomaly_labels = iso_forest.fit_predict(X_scaled)
    anomaly_scores = -iso_forest.score_samples(X_scaled)
    return anomaly_labels, anomaly_scores

anomaly_labels, anomaly_scores = detect_anomalies(df)
df['is_anomaly'] = anomaly_labels
df['anomaly_score'] = anomaly_scores

print(f"Anomalies detected: {(anomaly_labels == -1).sum()} / {len(df)}")
print(f"Anomaly scores: {anomaly_scores}")

## Cartel Likelihood Scoring

Combine anomaly and domain-specific features.

In [ ]:
def compute_cartel_likelihood(df, anomaly_scores):
    scores = []
    for i, row in df.iterrows():
        anom_norm = (anomaly_scores[i] - anomaly_scores.min()) / (anomaly_scores.max() - anomaly_scores.min() + 1e-8)
        mutual_signal = min(row['mutual_citation_ratio'] / MUTUAL_CITATION_RATIO_THRESHOLD, 1.0)
        partners_signal = min(row['n_mutual_partners'] / N_MUTUAL_PARTNERS_THRESHOLD, 1.0)
        cartel_score = 0.5 * anom_norm + 0.25 * mutual_signal + 0.25 * partners_signal
        scores.append(cartel_score)
    return np.array(scores)

cartel_scores = compute_cartel_likelihood(df, anomaly_scores)
df['cartel_score'] = cartel_scores

df_ranked = df.sort_values('cartel_score', ascending=False)
print("\nTop suspected cartels:")
print(df_ranked[['journal_name', 'cartel_score', 'mutual_citation_ratio', 'n_mutual_partners', 'label']])

## Results & Validation

Validate against ground truth labels.

In [ ]:
print("\n=== VALIDATION ===\n")
print(f"Ground truth labels: {df['label'].value_counts().to_dict()}")
print(f"Detected anomalies: {(df['is_anomaly'] == -1).sum()} journals")

cartel_indices = df[df['label'] != 'normal'].index
if len(cartel_indices) > 0:
    detected = (df.loc[cartel_indices, 'is_anomaly'] == -1).sum()
    print(f"\nGround-truth cartels: {len(cartel_indices)}")
    print(f"Detected by method: {detected} / {len(cartel_indices)}")
else:
    print(f"\nNo cartel examples in sample of {len(df)} journals")

print(f"\n=== FEATURE SUMMARY ===\n")
print(df[['journal_name', 'in_degree', 'out_degree', 'mutual_citation_weight', 'cartel_score', 'label']].to_string())

## Visualization

Plot cartel scores and network features.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Citation Cartel Detection Results', fontsize=14, fontweight='bold')

ax = axes[0, 0]
journals_display = df['journal_name'].str[:20]
colors = ['red' if label != 'normal' else 'blue' for label in df['label']]
ax.barh(journals_display, df['cartel_score'], color=colors, alpha=0.7)
ax.set_xlabel('Cartel Likelihood Score')
ax.set_title('Cartel Detection Scores')

ax = axes[0, 1]
ax.scatter(df['n_mutual_partners'], df['mutual_citation_ratio'], s=100, c=df['cartel_score'], cmap='RdYlBu_r', alpha=0.7)
ax.set_xlabel('Number of Mutual Partners')
ax.set_ylabel('Mutual Citation Ratio')
ax.set_title('Mutual Citation Patterns')

ax = axes[1, 0]
ax.scatter(df['in_degree'], df['out_degree'], s=100, c=df['cartel_score'], cmap='RdYlBu_r', alpha=0.7)
ax.set_xlabel('In-Degree')
ax.set_ylabel('Out-Degree')
ax.set_title('Citation Asymmetry')
for i, name in enumerate(df['journal_name']):
    ax.annotate(name[:10], (df['in_degree'].iloc[i], df['out_degree'].iloc[i]), fontsize=8)

ax = axes[1, 1]
ax.hist(df[df['is_anomaly'] == 1]['anomaly_score'], bins=5, alpha=0.5, label='Normal', color='blue')
if (df['is_anomaly'] == -1).sum() > 0:
    ax.hist(df[df['is_anomaly'] == -1]['anomaly_score'], bins=5, alpha=0.5, label='Anomaly', color='red')
ax.set_xlabel('Anomaly Score')
ax.set_ylabel('Frequency')
ax.set_title('Anomaly Score Distribution')
ax.legend()

plt.tight_layout()
plt.savefig('cartel_detection_results.png', dpi=100, bbox_inches='tight')
plt.show()
print("Visualization saved to cartel_detection_results.png")

## Summary

The method combines:
1. **Hodge Decomposition** — Separates asymmetric and mutual citation flows
2. **Louvain Community Detection** — Identifies clusters of journals
3. **Isolation Forest** — Flags anomalous network patterns

**Key insight:** Citation cartels exhibit high mutual citation ratios, many mutual partners, and anomalous network patterns.